# Équinoxe 2026 rent increase · CodeML 2026 · JADCO "Clés en main"

**Question:** how much will Collection Équinoxe's rents go up in 2026?

**Short version:** we measure the *annualised growth of effective rent for the same apartment*
(each apartment compared with its own previous lease), separately for renewals and new tenants,
in Québec and in Ontario. We then forecast it for 2026 by reconciling our internal trend with public anchors
(TAL, Statistics Canada rent CPI), and we check the method with a backtest on 2023, 2024 and 2025.

| § | Contents |
|---|---|
| 0 | Libraries, constants, loading and renaming |
| 1 | Definition of the increase we measure |
| 2 | Data audit and data dictionary |
| 3 | Mix effect (why the median lies) |
| 4 | Same-apartment pairs |
| 5 | Concessions: contract vs effective rent |
| 6 | Renewals vs new tenants, QC / ON rules |
| 7 | External data and reconciliation |
| 8 | Forecast `estimate_2026()` |
| 9 | Backtest `backtest()` 2023–2025 |
| 10 | Results, scenarios, building detail, references |

> **Confidentiality:** the CRM extract is never shown row by row; only aggregates appear.

## 0. Setup

### 0.1 Libraries and project location
Imports, plus a small helper that finds the project root (the folder with `data/raw/` in it). We need it because the same
code runs both from `nb/` (one notebook per section) and from the root (the assembled final notebook).

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
plt.rcParams.update({"lines.linewidth": 2, "axes.grid": True, "grid.alpha": 0.3})  # shared chart style


def find_project_root(start=None):
    """Walk up from the current folder to the first folder that contains data/raw/."""
    here = Path(start or Path.cwd()).resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "data" / "raw").is_dir():
            return candidate
    raise FileNotFoundError("data/raw/ not found: copy the 4 Équinoxe CSV files into data/raw/ (see README).")


PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
EXTERNAL_DIR = PROJECT_ROOT / "data" / "external"
MODELS_DIR = PROJECT_ROOT / "models"
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

### 0.2 Constants
Every judgement call we made (years, thresholds, weights) lives here, named once, with a comment saying why.
Nothing further down uses a bare number, so if you disagree with a threshold, this is the one cell to read.

In [2]:
# --- Dates ---
DATA_CUTOFF = pd.Timestamp("2025-12-31")  # CRM extraction date: no data after it
FORECAST_YEAR = 2026                      # year to forecast
BACKTEST_YEARS = (2023, 2024, 2025)       # validation years required by the brief
FIRST_ANALYSIS_YEAR = 2018                # first leases in 2017 -> first pairs in 2018
DAYS_PER_YEAR = 365.25                    # average year (leap years included)
CPI_LAST_KNOWN_MONTH = 11                 # November CPI comes out mid-December: last month known on Dec 31

# --- Unit key and pairing ---
UNIT_KEY = ["prop_code", "unit_code"]     # = sPropCode + sUnitCode. NEVER sSite + sUnitCode (collisions, §2)
MIN_GAP_YEARS = 0.5                       # shorter gap = same lease entered twice, or a short lease that isn't comparable
MAX_GAP_YEARS = 2.5                       # longer gap = the change mixes several years
GAP_DISPLAY_EDGES = (0.9, 1.1, 1.5)       # display only (§4.2): ≈ 1 year, a bit more, clearly more
GAP_SENSITIVITY_BOUNDS = [(0.75, 1.5), (0.9, 1.1), (0.5, 2.5)]  # alternative bounds tested in §4.6

# --- Jurisdictions and lease types ---
PROVINCE_CODE = {"Quebec": "QC", "Ontario": "ON"}      # sState labels -> short codes
LEASE_TYPE_LABEL = {1: "renewal", 0: "turnover"}        # is_renewal -> label (turnover = new tenant)
SEGMENT_COLS = ["province", "lease_type"]               # the model's 4 segments
LARGE_UNIT_MIN_BEDROOMS = 2                             # "large unit" = 2 bedrooms or more (§3)
SECTION_F_WINDOW_YEARS = 5                              # QC: building ready for habitation for <= 5 years (section F of the lease)
ONTARIO_EXEMPTION_DATE = pd.Timestamp("2018-11-15")     # ON: first occupied after this date = exempt from the guideline

# --- Forecast ---
W_INTERNAL = 0.5                 # weight of the internal trend vs the external anchor, fixed beforehand (no reason to favour either)
TREND_WINDOW_YEARS = 3           # last 3 years: smooths the noise without going back before the 2022 inflation shock
MIN_PAIRS_PER_SEGMENT_YEAR = 20  # below this many pairs, a yearly median is too noisy for the trend
DEPTH_PERSISTENCE_CENTRAL = 0.5  # share of last year's deepening of concessions that repeats (neutral between 0 and 1)
DEPTH_PERSISTENCE_LOW = 1.0      # low scenario: concessions keep deepening at the same pace
DEPTH_PERSISTENCE_HIGH = 0.0     # high scenario: concession depth stops growing
TURNOVER_ANCHOR_SOURCE = "cpi"   # anchor for QC turnovers and Ontario: "cpi" (rent CPI) or "cmhc" (CMHC, compared in §9.4)
CMHC_GEO_BY_PROVINCE = {"QC": "Montreal", "ON": "Ottawa"}  # reference CMHC metro area for each province
MIN_PAIRS_PER_CELL = 15          # building x bedrooms bonus: below this, fall back on the segment
N_BOOTSTRAP = 1000               # resamples for the uncertainty interval
BOOTSTRAP_QUANTILES = (0.10, 0.90)  # 80 % interval
RANDOM_SEED = 2026               # reproducibility
SENSITIVITY_W_VALUES = (0.0, 0.25, 0.5, 0.75, 1.0)  # grid read in §9.4 (not used for tuning)
SENSITIVITY_WINDOWS = (1, 2, 3)                     # trend windows tested in §9.4

# --- Display ---
PCT = 100                        # fraction -> percentage (display only; we store fractions)
FIGSIZE = (9, 4.5)
WIDE_FIGSIZE = (12.6, 4.5)       # two charts side by side
NUMERIC_TOLERANCE = 1e-9        # rounding tolerance (rent comparisons, depth < 1)
SOURCE_CRM = "Source: Yardi CRM extract, Équinoxe, as of 2025-12-31; team calculations"
LEASE_TYPE_CHART_LABELS = {"renewal": "renewals", "turnover": "new tenants"}  # chart legends
DASHBOARD_PATH = PROJECT_ROOT / "dashboard" / "equinoxe_dashboard.html"  # dashboard (bonus, §10)


def add_source(fig, text=SOURCE_CRM):
    """Add the source line under a chart."""
    fig.text(0.01, -0.02, text, fontsize="small")

### 0.3 Loading the four files
We read the four CSVs from `data/raw/` exactly as delivered (we never modify or commit them).
What we should get: 1,061 units, 4,302 leases, 3,560 concession lines, 3,857 advertised months.

In [3]:
raw_listings = pd.read_csv(RAW_DIR / "equinoxe_listings.csv")
raw_leases = pd.read_csv(RAW_DIR / "equinoxe_lease_history.csv")
raw_concessions = pd.read_csv(RAW_DIR / "equinoxe_concessions.csv")
raw_asking = pd.read_csv(RAW_DIR / "equinoxe_asking_history.csv")

for table_name, table in [("listings", raw_listings), ("leases", raw_leases),
                          ("concessions", raw_concessions), ("asking", raw_asking)]:
    print(f"{table_name:<12} {len(table):>6,} rows  {table.shape[1]:>2} columns")

listings      1,061 rows  29 columns
leases        4,302 rows  35 columns
concessions   3,560 rows  18 columns
asking        3,857 rows  14 columns


### 0.4 Renaming columns after what they really contain
We swap the Yardi names (`sRent`, `sRenewal`…) for names that describe what the columns *actually* contain
(see the dictionary in §2). The sneaky one: `sRent` does not mean the same thing in the two files (current advertised rent in
`listings`, rent written on the lease in `leases`), so it gets two different names. `PromoPay` becomes `one_time_promo_credit`
because it is a one-off credit, not a monthly charge (shown in §2).
Honest names now save us from misreading things later.

**The h / s prefixes (Yardi convention):** columns starting with **h** are *handles*, internal IDs used only to join tables (`hUnit`, `hProperty`, `hBuilding`). Columns starting with **s** are *stored values*, what a person reads and what we group by (`sRent`, `sBuilding`, `sPropCode`). So we join on `h*` columns (now `unit_id`) and label or group with `s*` columns (now `building`, `prop_code`). The prefix says how a column is used, not its type.

In [4]:
SHARED_RENAME = {
    "hUnit": "unit_id", "hProperty": "property_id", "hBuilding": "building_id",
    "sPropCode": "prop_code", "sUnitCode": "unit_code", "sBuilding": "building",
    "sCity": "city", "sState": "province_name", "sSite": "site_slug", "sLink": "unit_link",
    "sAddr1": "address", "sLatitude": "latitude", "sLongitude": "longitude",
    "sBeds": "bedrooms", "sBaths": "bathrooms", "sSqft": "sqft", "sFloor": "floor",
    "sUnitType": "layout_code", "sUnitSubtype": "bedroom_label", "sLeaseTerm": "term_label",
    "sPropType": "property_type", "sFurnishing": "furnishing", "sSmoking": "smoking",
    "sCats": "cats_allowed", "sDogs": "dogs_allowed", "sUnitDesc": "unit_description",
}
LEASE_RENAME = {
    "sRent": "rent_contract", "sRentEffective": "rent_effective", "sConcession": "has_concession",
    "sRenewal": "is_renewal", "sTermSeq": "lease_seq", "sTermMonths": "term_months",
    "sSignDate": "sign_date", "sLeaseFrom": "lease_start", "sLeaseTo": "lease_end",
    "sAvailable": "available_date",
}
LISTING_RENAME = {"sRent": "asking_rent_current", "sAsOf": "asking_as_of", "sAvailable": "available_date"}
CONCESSION_RENAME = {
    "sChargeCode": "concession_code", "sChargeDesc": "concession_desc", "sAmount": "concession_amount",
    "sMonths": "concession_months", "sDateFrom": "concession_start", "sDateTo": "concession_end",
}
ASKING_RENAME = {"sAskingRent": "asking_rent", "sMonth": "asking_month"}
CONCESSION_CODE_RENAME = {"PromoPay": "one_time_promo_credit"}  # one-off credit, not monthly (§2.6)

units = raw_listings.rename(columns={**SHARED_RENAME, **LISTING_RENAME})
leases = raw_leases.rename(columns={**SHARED_RENAME, **LEASE_RENAME})
concessions = raw_concessions.rename(columns={**SHARED_RENAME, **CONCESSION_RENAME})
asking = raw_asking.rename(columns={**SHARED_RENAME, **ASKING_RENAME})
concessions["concession_code"] = concessions["concession_code"].replace(CONCESSION_CODE_RENAME)

### 0.5 Types, derived columns and dropped columns
Three housekeeping steps:
1. dates converted to `datetime`; `asking_month` ("YYYY-MM") becomes a date on the 1st of the month;
2. derived columns: `province` (QC/ON), `lease_year` (year the lease starts = the year a lease is attributed to),
   `lease_type` (renewal/turnover), `concession_depth = 1 − effective rent / contract rent`;
3. we drop `available_date` from `leases` (identical to `lease_start` on 100 % of rows, checked below) and
   columns with a single value (no information).

Net effect: same number of rows, a few columns gone, four new ones.

In [5]:
for date_col in ["sign_date", "lease_start", "lease_end", "available_date"]:
    leases[date_col] = pd.to_datetime(leases[date_col])
for date_col in ["concession_start", "concession_end"]:
    concessions[date_col] = pd.to_datetime(concessions[date_col])
asking["asking_date"] = pd.to_datetime(asking["asking_month"] + "-01")

assert (leases["available_date"] == leases["lease_start"]).all(), "available_date is no longer redundant"
constant_cols = [c for c in leases.columns if leases[c].nunique() == 1]
leases = leases.drop(columns=["available_date", *constant_cols])
print("Columns dropped from leases:", ["available_date", *constant_cols])

for table in (units, leases, concessions, asking):
    table["province"] = table["province_name"].map(PROVINCE_CODE)
    assert table["province"].notna().all(), "unknown province"

leases["lease_year"] = leases["lease_start"].dt.year
leases["lease_type"] = leases["is_renewal"].map(LEASE_TYPE_LABEL)
leases["concession_depth"] = 1 - leases["rent_effective"] / leases["rent_contract"]
asking["asking_year"] = asking["asking_date"].dt.year
print(f"leases: {leases.shape[0]:,} rows, {leases.shape[1]} columns")

Columns dropped from leases: ['available_date', 'property_type', 'furnishing', 'smoking', 'cats_allowed', 'dogs_allowed']
leases: 4,302 rows, 33 columns


In [6]:
units.to_parquet(INTERIM_DIR / "units.parquet")
leases.to_parquet(INTERIM_DIR / "leases.parquet")
concessions.to_parquet(INTERIM_DIR / "concessions.parquet")
asking.to_parquet(INTERIM_DIR / "asking.parquet")